# Frozen two-stage cascade
Uses the approved LightGBM and GraphSAGE checkpoints. Calibrates on the complete validation split, then evaluates the full test split.


In [ ]:
from pathlib import Path
import os, sys, tarfile

INPUT = Path('/kaggle/input')
WORK = Path('/kaggle/working')
WORK.mkdir(parents=True, exist_ok=True)
bundle = INPUT / 'datasets/nhatnguyen2101/optc-cascade-bundle'
required_bundle = ('bundle_manifest.json', 'provenance/graph.sqlite', 'provenance/model_numpy.npz', 'models/lgbm_model.txt', 'code/cascade_runtime.py')
if not all((bundle / name).is_file() for name in required_bundle):
    bundles = sorted(INPUT.glob('**/provenance-cascade-bundle-v2.tar.gz'))
    if len(bundles) != 1:
        raise RuntimeError(f'Expected an extracted cascade bundle or one archive; found {len(bundles)} archives')
    bundle = WORK / 'cascade_bundle_v2'
    bundle.mkdir(exist_ok=True)
    with tarfile.open(bundles[0], 'r:gz') as tar:
        for member in tar.getmembers():
            target = (bundle / member.name).resolve()
            if not target.is_relative_to(bundle.resolve()) or not member.isfile():
                raise ValueError(f'Unsafe bundle member: {member.name}')
        tar.extractall(bundle)
sys.path[:0] = [str(bundle / 'code')]
from cascade_runtime import run

# Host shards may be attached as an uncompressed dataset or as host-input.tar.
host_root = INPUT / 'datasets/nhatnguyen2101/optc-host-ml-full-input'
shard_manifests = [host_root / 'manifest.json'] if (host_root / 'manifest.json').is_file() and (host_root / 'assets/vocab.json').is_file() else []
if not shard_manifests:
    shard_manifests = [p for p in INPUT.glob('**/manifest.json') if (p.parent / 'assets/vocab.json').is_file()]
if not shard_manifests:
    host_archives = sorted(INPUT.glob('**/host-input.tar'))
    if len(host_archives) != 1:
        raise RuntimeError(f'Expected one host-input.tar when no extracted shards exist; found {len(host_archives)}')
    host_dir = WORK / 'host_shards'
    host_dir.mkdir(exist_ok=True)
    with tarfile.open(host_archives[0], 'r') as tar:
        for member in tar.getmembers():
            target = (host_dir / member.name).resolve()
            if not target.is_relative_to(host_dir.resolve()) or not member.isfile():
                raise ValueError(f'Unsafe host archive member: {member.name}')
        tar.extractall(host_dir)
    shard_manifests = [host_dir / 'manifest.json']
candidates = []
if os.environ.get('OPTC_MANIFEST'):
    candidates.append(Path(os.environ['OPTC_MANIFEST']))
else:
    for p in shard_manifests:
        try:
            import json
            d = json.loads(p.read_text())
            if 'shards' in d and 'vocab_size' in d: candidates.append(p)
        except (OSError, ValueError): pass
if len(candidates) != 1:
    raise RuntimeError(f'Expected one host shard manifest; found {len(candidates)}. Set OPTC_MANIFEST.')
manifest = candidates[0]
print('Bundle:', bundle)
print('Host manifest:', manifest)
print('Output:', WORK / 'cascade_run_v2')
result = run(manifest, bundle, WORK / 'cascade_run_v2')
print('Download:', WORK / 'cascade_run_v2/optc-two-stage-cascade-results.zip')
